### RAG_PIPELINE- data ingestion to vector db pipeline

In [71]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

### Data Ingestion (yaha se ham data bharte hai)

In [72]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: objectdetection.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/BWUBRD+CMBX12', '/Encoding': IndirectObject(484, 0, 13070343760), '/FirstChar': 12, '/FontDescriptor': IndirectObject(485, 0, 13070343760), '/LastChar': 122, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [625, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 437, 437, 0, 0, 0, 375, 312, 562, 562, 562, 562, 562, 562, 562, 562, 562, 562, 562, 0, 0, 0, 0, 0, 0, 0, 849, 799, 812, 862, 738, 707, 884, 879, 418, 0, 880, 675, 1067, 879, 844, 768, 844, 839, 625, 782, 864, 0, 1162, 0, 0, 687, 0, 0, 0, 0, 0, 0, 546, 625, 500, 625, 513, 343, 562, 625, 312, 0, 593, 312, 937, 625, 562, 625, 593, 459, 443, 437, 625, 593, 812, 593, 593, 500]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/FTYRXR+SFRM1000', '/Encoding'

  ✓ Loaded 88 pages

Processing: embeddings.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/BEXNDE+CMR12', '/Encoding': IndirectObject(495, 0, 13070343760), '/FirstChar': 11, '/FontDescriptor': IndirectObject(496, 0, 13070343760), '/LastChar': 127, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [571, 543, 543, 815, 0, 271, 0, 0, 489, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 489, 0, 0, 815, 761, 271, 380, 380, 489, 761, 271, 326, 271, 489, 489, 489, 489, 489, 489, 489, 489, 489, 489, 489, 271, 271, 0, 761, 0, 462, 0, 734, 693, 707, 747, 666, 639, 768, 734, 353, 503, 761, 611, 897, 734, 761, 666, 761, 720, 543, 707, 734, 734, 1006, 734, 734, 598, 271, 0, 271, 0, 0, 0, 489, 543, 435, 543, 435, 299, 489, 543, 271, 299, 516, 271, 815, 543, 489, 543, 516, 380, 386, 380, 543, 516, 707, 516, 516, 435, 489, 979, 0, 0, 489]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 f

  ✓ Loaded 27 pages

Processing: attention.pdf
  ✓ Loaded 11 pages

Processing: proposal.pdf
  ✓ Loaded 8 pages

Total documents loaded: 134


In [ ]:
all_pdf_documents 

[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf_files/objectdetection.pdf', 'total_pages': 88, 'page': 0, 'page_label': '1', 'source_file': 'objectdetection.pdf', 'file_type': 'pdf'}, page_content='Object Detection\nSihao Liang\nJiajun Lu\nKevin Perkins'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf_files/objectdetection.pdf', 'total_pages': 88, 'page': 1, 'page_label': '2', 'source_file': 'objectdetection.pdf', 'file_type': 'pdf'}, page_content='Overview\nIntro\nPart I: Two Stage Detection\nPart II: Unified Detection\nPart III: Others\nSummary and comparison'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf_files/objectdetection.pdf', 'total_pages': 88, 'page': 2, 'page_label': '3', 'source_file': 'objectdetection.pdf', 'file_type': 'pdf'}, page_content='What  is object detection\nhttp://cs231n.stanford.edu/slid

### chunking (yaha pe data ko todte hai tukdo mei taki aage usse uske hisab se convert krpaye)

In [74]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [75]:
chunks=split_documents(all_pdf_documents)
chunks

Split 134 documents into 249 chunks

Example chunk:
Content: Object Detection
Sihao Liang
Jiajun Lu
Kevin Perkins...
Metadata: {'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf_files/objectdetection.pdf', 'total_pages': 88, 'page': 0, 'page_label': '1', 'source_file': 'objectdetection.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf_files/objectdetection.pdf', 'total_pages': 88, 'page': 0, 'page_label': '1', 'source_file': 'objectdetection.pdf', 'file_type': 'pdf'}, page_content='Object Detection\nSihao Liang\nJiajun Lu\nKevin Perkins'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf_files/objectdetection.pdf', 'total_pages': 88, 'page': 1, 'page_label': '2', 'source_file': 'objectdetection.pdf', 'file_type': 'pdf'}, page_content='Overview\nIntro\nPart I: Two Stage Detection\nPart II: Unified Detection\nPart III: Others\nSummary and comparison'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf_files/objectdetection.pdf', 'total_pages': 88, 'page': 2, 'page_label': '3', 'source_file': 'objectdetection.pdf', 'file_type': 'pdf'}, page_content='What  is object detection\nhttp://cs231n.stanford.edu/slid

### embedding (idhr tuta hua data vectors mei banjata hai )

In [76]:
import numpy as np
from sentence_transformers import SentenceTransformer #for embedding
import chromadb
from chromadb.config import Settings
import uuid # for storing id of stored data
from typing import List,Dict,Tuple, Any
from sklearn.metrics.pairwise import cosine_similarity #for retreival part

In [77]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7157.28it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/4z/7p_db1bn6b580mwk7j36rr_c0000gn/T/ipykernel_34563/2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore (sara vector mei bana hua data idhr aake store hota h)

In [78]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
              """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 308


In [79]:
chunks


[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf_files/objectdetection.pdf', 'total_pages': 88, 'page': 0, 'page_label': '1', 'source_file': 'objectdetection.pdf', 'file_type': 'pdf'}, page_content='Object Detection\nSihao Liang\nJiajun Lu\nKevin Perkins'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf_files/objectdetection.pdf', 'total_pages': 88, 'page': 1, 'page_label': '2', 'source_file': 'objectdetection.pdf', 'file_type': 'pdf'}, page_content='Overview\nIntro\nPart I: Two Stage Detection\nPart II: Unified Detection\nPart III: Others\nSummary and comparison'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf_files/objectdetection.pdf', 'total_pages': 88, 'page': 2, 'page_label': '3', 'source_file': 'objectdetection.pdf', 'file_type': 'pdf'}, page_content='What  is object detection\nhttp://cs231n.stanford.edu/slid

In [80]:
### convert text to embeddings
texts = [doc.page_content for doc in chunks]
### generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)
### store in vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 249 texts...


Batches: 100%|██████████| 8/8 [00:00<00:00, 10.83it/s]


Generated embeddings with shape: (249, 384)
Adding 249 documents to vector store...
Successfully added 249 documents to vector store
Total documents in collection: 557


### Rag Retreival Pipeline 

from here main work starts, here we generate context that will be suplied to llm
isme ek query aegi, vo query bhi embed hoegi(vector), then available vecDb se compare hogi,
uss basis pr context banega jo llm mei further jaega

In [81]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'") 
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [82]:
rag_retriever

In [83]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.24it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

### llm integration

here we integrate llm, using groq as free, we feed it context and a prompt

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [85]:
answer=rag_simple("What is attention mechanism?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 135.76it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
No relevant context found to answer the question.
